In [ ]:
# ============================================
# CELLULE 1 — Vérification des dépendances
# ============================================

from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn
import xgboost

# Connexion à la session Snowflake
session = get_active_session()

print("✅ snowpark   : OK")
print(f"✅ pandas     : {pd.__version__}")
print(f"✅ numpy      : {np.__version__}")
print(f"✅ sklearn    : {sklearn.__version__}")
print(f"✅ matplotlib : {matplotlib.__version__}")
print(f"✅ seaborn    : {seaborn.__version__}")
print(f"✅ xgboost    : {xgboost.__version__}")
print(f"\n🔗 Connecté à Snowflake : {session.get_current_database()}.{session.get_current_schema()}")
print("\n🎉 Environnement prêt !")





In [ ]:
sql_statements = [
    """
    CREATE OR REPLACE FILE FORMAT HOUSE_JSON_FORMAT
      TYPE = 'JSON'
      STRIP_OUTER_ARRAY = TRUE
    """,
    """
    CREATE OR REPLACE STAGE HOUSE_STAGE
      URL = 's3://logbrain-datalake/datasets/house_price/'
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """,
    """
    CREATE OR REPLACE TABLE HOUSE_PRICES_RAW (
      raw_data VARIANT
    )
    """,
    """
    COPY INTO HOUSE_PRICES_RAW
      FROM @HOUSE_STAGE/Housing_Price_Data.json
      FILE_FORMAT = HOUSE_JSON_FORMAT
    """
 ]

for stmt in sql_statements:
    session.sql(stmt).collect()

print("✅ Ingestion JSON terminée")
session.sql("SELECT COUNT(*) AS nb_lignes FROM HOUSE_PRICES_RAW").show()
session.sql("SELECT raw_data FROM HOUSE_PRICES_RAW LIMIT 3").show()

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE HOUSE_PRICES AS
SELECT
  raw_data:price::NUMBER             AS price,
  raw_data:area::NUMBER              AS area,
  raw_data:bedrooms::NUMBER          AS bedrooms,
  raw_data:bathrooms::NUMBER         AS bathrooms,
  raw_data:stories::NUMBER           AS stories,
  raw_data:mainroad::VARCHAR         AS mainroad,
  raw_data:guestroom::VARCHAR        AS guestroom,
  raw_data:basement::VARCHAR         AS basement,
  raw_data:hotwaterheating::VARCHAR  AS hotwaterheating,
  raw_data:airconditioning::VARCHAR  AS airconditioning,
  raw_data:parking::NUMBER           AS parking,
  raw_data:prefarea::VARCHAR         AS prefarea,
  raw_data:furnishingstatus::VARCHAR AS furnishingstatus
FROM HOUSE_PRICES_RAW
""").collect()

print("✅ Table HOUSE_PRICES créée")
session.sql("SELECT COUNT(*) AS nb_lignes FROM HOUSE_PRICES").show()
session.sql("SELECT * FROM HOUSE_PRICES LIMIT 5").show()

In [ ]:
# ============================================
# CELLULE 4 — Chargement & exploration
# ============================================

# Charger la table en DataFrame Snowpark
df = session.table("HOUSE_PRICES")

print(f"📊 Nombre de lignes  : {df.count()}")
print(f"📋 Nombre de colonnes: {len(df.columns)}")
print(f"\n📌 Colonnes : {df.columns}")

In [ ]:
# ============================================
# CELLULE 5 — Aperçu des données
# ============================================

# Aperçu des 5 premières lignes
print("=== 5 premières lignes ===")
df.show(5)

# Statistiques descriptives
print("\n=== Statistiques descriptives ===")
df.describe().show()

In [ ]:
# ============================================
# CELLULE 6 — Valeurs manquantes
# ============================================
from snowflake.snowpark.functions import col, count, when

# Compter les nulls par colonne
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

print("=== Valeurs manquantes par colonne ===")
null_counts.show()

In [ ]:
# ============================================
# CELLULE 7 — Distribution du prix
# ============================================
import matplotlib.pyplot as plt

# Convertir en Pandas pour visualiser
df_pd = df.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(df_pd['PRICE'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des Prix')
axes[0].set_xlabel('Prix')
axes[0].set_ylabel('Fréquence')

# Boxplot
axes[1].boxplot(df_pd['PRICE'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Boxplot des Prix')
axes[1].set_ylabel('Prix')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 8 — Matrice de corrélation
# ============================================
import seaborn as sns

# Sélectionner uniquement les colonnes numériques
num_cols = ['PRICE', 'AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']
corr_matrix = df_pd[num_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Matrice de Corrélation')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 9 — Variables catégorielles
# ============================================

cat_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
            'HOTWATERHEATING', 'AIRCONDITIONING',
            'PREFAREA', 'FURNISHINGSTATUS']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col_name in enumerate(cat_cols):
    counts = df_pd[col_name].value_counts()
    axes[i].bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col_name}')
    axes[i].set_ylabel('Count')

# Masquer le dernier subplot vide
axes[-1].set_visible(False)

plt.suptitle('Distribution des Variables Catégorielles', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 10 — Encodage des variables
# ============================================
import pandas as pd

df_pd = df.to_pandas()

# Colonnes binaires (yes/no) → 1/0
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
               'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA']

for col_name in binary_cols:
    df_pd[col_name] = df_pd[col_name].map({'yes': 1, 'no': 0})

# Colonne furnishingstatus → encodage ordinal
df_pd['FURNISHINGSTATUS'] = df_pd['FURNISHINGSTATUS'].map({
    'furnished': 2,
    'semi-furnished': 1,
    'unfurnished': 0
})

print("✅ Encodage terminé !")
print(df_pd.head())
print(f"\nTypes :\n{df_pd.dtypes}")

In [ ]:
# ============================================
# CELLULE 11 — Séparation features / cible
# ============================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Variable cible
y = df_pd['PRICE']

# Features (toutes les colonnes sauf PRICE)
X = df_pd.drop(columns=['PRICE'])

print(f"✅ Features X : {X.shape}")
print(f"✅ Cible y    : {y.shape}")
print(f"\n📌 Colonnes X : {list(X.columns)}")

In [ ]:
# ============================================
# CELLULE 12 — Train / Test split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42
)

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"✅ Taille X_train : {X_train_scaled.shape}")
print(f"✅ Taille X_test  : {X_test_scaled.shape}")
print(f"\n📊 Répartition :")
print(f"   Train : {len(X_train)} lignes ({len(X_train)/len(X)*100:.0f}%)")
print(f"   Test  : {len(X_test)} lignes  ({len(X_test)/len(X)*100:.0f}%)")

In [ ]:
# ============================================
# CELLULE 13 — Entraînement des modèles
# ============================================
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Définir les modèles
models = {
    "Linear Regression" : LinearRegression(),
    "Random Forest"     : RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost"           : XGBRegressor(n_estimators=100, random_state=42)
}

# Entraîner chaque modèle
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"✅ {name} entraîné !")

print("\n🎉 Tous les modèles sont entraînés !")

In [ ]:
# ============================================
# CELLULE 14 — Évaluation des modèles
# ============================================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    
    results.append({
        "Modèle" : name,
        "MAE"    : round(mae, 2),
        "RMSE"   : round(rmse, 2),
        "R²"     : round(r2, 4)
    })
    print(f"📊 {name}")
    print(f"   MAE  : {mae:,.0f}")
    print(f"   RMSE : {rmse:,.0f}")
    print(f"   R²   : {r2:.4f}\n")

# Tableau récapitulatif
results_df = pd.DataFrame(results)
print("=== Tableau comparatif ===")
print(results_df.to_string(index=False))

In [ ]:
# ============================================
# CELLULE 15 — Visualisation des performances
# ============================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['MAE', 'RMSE', 'R²']
colors  = ['#e74c3c', '#e67e22', '#2ecc71']

for i, metric in enumerate(metrics):
    axes[i].bar(results_df['Modèle'], results_df[metric],
                color=colors[i], edgecolor='white')
    axes[i].set_title(f'Comparaison — {metric}')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Comparaison des Modèles ML', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# CELLULE 16 — Optimisation avec GridSearchCV (pipeline complet)
# ============================================
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Paramètres réduits pour éviter la surcharge mémoire
param_grid = {
    'xgb__n_estimators': [100, 200],
    'xgb__max_depth': [3, 5],
    'xgb__learning_rate': [0.1, 0.2],
}

pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(random_state=42))
])

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1,
    verbose=1
)

print("⏳ GridSearch en cours...")
grid_search.fit(X_train, y_train)

print(f"\n✅ Meilleurs paramètres : {grid_search.best_params_}")
print(f"✅ Meilleur R² (CV)     : {grid_search.best_score_:.4f}")

In [ ]:
# ============================================
# CELLULE 17 — Évaluation du meilleur modèle
# ============================================

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

mae_best  = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best   = r2_score(y_test, y_pred_best)

print("=== Modèle XGBoost Optimisé (Pipeline) ===")
print(f"   MAE  : {mae_best:,.0f}")
print(f"   RMSE : {rmse_best:,.0f}")
print(f"   R²   : {r2_best:.4f}")

# Comparaison avant/après
xgb_base = trained_models['XGBoost']
y_pred_base = xgb_base.predict(X_test_scaled)
r2_base = r2_score(y_test, y_pred_base)

print(f"\n📊 Comparaison :")
print(f"   XGBoost de base   → R² : {r2_base:.4f}")
print(f"   XGBoost optimisé  → R² : {r2_best:.4f}")
improvement = (r2_best - r2_base) * 100
print(f"   Amélioration      : +{improvement:.2f}%")

In [ ]:
# ============================================
# CELLULE 18 — Prédictions vs Réalité
# ============================================

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue', edgecolors='white')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Prédiction parfaite')
plt.xlabel('Prix Réel')
plt.ylabel('Prix Prédit')
plt.title('XGBoost Optimisé — Prédictions vs Réalité')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 19 — Sauvegarde dans le Model Registry
# ============================================
from snowflake.ml.registry import Registry

# Créer le registry
registry = Registry(session=session)

# Enregistrer le meilleur modèle (pipeline complet: scaler + XGBoost)
model_ref = registry.log_model(
    model=best_model,
    model_name="house_price_xgboost",
    version_name="v1",
    comment="Pipeline XGBoost optimisé avec GridSearch",
    metrics={
        "r2": round(r2_best, 4),
        "mae": round(mae_best, 2),
        "rmse": round(rmse_best, 2)
    },
    sample_input_data=X_test.head(5)
)

print("✅ Modèle enregistré dans le Snowflake Model Registry !")
print("   Nom    : house_price_xgboost")
print("   Version: v1")
print(f"   R²     : {r2_best:.4f}")

In [ ]:
# ============================================
# CELLULE 22 — Vérification du registry
# ============================================

# Lister les modèles disponibles
models_list = registry.show_models()
print("=== Modèles dans le Registry ===")
print(models_list)

In [ ]:
# ============================================
# CELLULE 20 — Chargement du modèle depuis le Registry
# ============================================
from snowflake.ml.registry import Registry
import pandas as pd

# Charger le registry
registry = Registry(session=session)

# Récupérer le modèle
model_ref = registry.get_model("house_price_xgboost").version("v1")

print("✅ Modèle chargé depuis le registry !")

In [ ]:
# ============================================
# CELLULE 21 — Inférence corrigée
# ============================================

# Créer quelques nouvelles maisons à prédire
new_houses = pd.DataFrame({
    'AREA': [200, 150, 300],
    'BEDROOMS': [3, 2, 4],
    'BATHROOMS': [2, 1, 3],
    'STORIES': [2, 1, 3],
    'MAINROAD': [1, 1, 0],
    'GUESTROOM': [0, 0, 1],
    'BASEMENT': [1, 0, 1],
    'HOTWATERHEATING': [0, 0, 1],
    'AIRCONDITIONING': [1, 0, 1],
    'PARKING': [2, 1, 3],
    'PREFAREA': [1, 0, 1],
    'FURNISHINGSTATUS': [2, 0, 1]
}).astype('int16')

# Le modèle enregistré contient déjà le scaler
predictions = model_ref.run(new_houses, function_name="predict")

print("=== Prédictions de prix ===")
for i, pred in enumerate(predictions['output_feature_0']):
    print(f"   Maison {i+1} : {pred:,.0f} €")

In [ ]:
# ============================================
# CELLULE 23 — Stocker les résultats dans Snowflake
# ============================================

# Ajouter les prédictions au dataframe
new_houses['PREDICTED_PRICE'] = predictions['output_feature_0'].values

# Sauvegarder dans une table Snowflake
session.write_pandas(
    new_houses,
    table_name    = "HOUSE_PREDICTIONS",
    auto_create_table = True,
    overwrite     = True
)

print("✅ Prédictions sauvegardées dans la table HOUSE_PREDICTIONS !")

# Vérifier
session.table("HOUSE_PREDICTIONS").show()

In [ ]:
# ============================================
# CELLULE 24 - Application Streamlit 
# ============================================
import streamlit as st
import pandas as pd
from streamlit.errors import StreamlitSetPageConfigMustBeFirstCommandError
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

# In Snowflake, page config may already be set by the runtime. Keep this safe and non-blocking.
try:
    st.set_page_config(
        page_title="House Price Studio",
        page_icon="🏠",
        layout="wide",
        initial_sidebar_state="expanded",
    )
except StreamlitSetPageConfigMustBeFirstCommandError:
    pass

st.markdown(
    """
    <style>
:root {
  --bg: #f4f6f8;
  --ink: #0f172a;
  --muted: #475569;
  --card: #ffffff;
  --line: #dbe3ea;
  --brand: #0f766e;
  --brand-2: #f59e0b;
  --ok: #15803d;
  --result-bg-a: #ffffff;
  --result-bg-b: #f8fffd;
  --result-ink: #0f172a;
}

@media (prefers-color-scheme: dark) {
  :root {
    --bg: #0b1220;
    --ink: #e5edf8;
    --muted: #9fb0c7;
    --card: #111a2b;
    --line: #27354b;
    --result-bg-a: #13253a;
    --result-bg-b: #0f1d2f;
    --result-ink: #f8fbff;
  }
}

html[data-theme="dark"],
body[data-theme="dark"],
[data-theme="dark"] {
  --bg: #0b1220;
  --ink: #e5edf8;
  --muted: #9fb0c7;
  --card: #111a2b;
  --line: #27354b;
  --result-bg-a: #13253a;
  --result-bg-b: #0f1d2f;
  --result-ink: #f8fbff;
}

html, body, [class*="css"] {
  font-family: "Trebuchet MS", "Segoe UI", sans-serif;
}

section.main > div {
  background: radial-gradient(circle at 10% -20%, #dff7f1 0%, transparent 35%),
              radial-gradient(circle at 90% -30%, #fde8c8 0%, transparent 30%),
              var(--bg);
}

div.block-container {
  padding-top: 1.25rem;
  padding-bottom: 2rem;
  max-width: 1280px;
}

div[data-testid="stSidebar"] {
  background: linear-gradient(180deg, #0b1320 0%, #111b2e 100%);
}

div[data-testid="stSidebar"] * {
  color: #e2e8f0 !important;
}

.hero {
  background: linear-gradient(120deg, #0f766e 0%, #14b8a6 45%, #f59e0b 100%);
  color: #ffffff;
  border-radius: 20px;
  padding: 1.1rem 1.3rem;
  box-shadow: 0 10px 24px rgba(15, 118, 110, 0.28);
  margin-bottom: 0.9rem;
}

.panel {
  background: var(--card);
  border: 1px solid var(--line);
  border-radius: 16px;
  padding: 0.9rem 1rem;
  box-shadow: 0 8px 20px rgba(15, 23, 42, 0.06);
  margin-bottom: 0.8rem;
}

.panel, .panel-title {
  color: var(--ink);
}

.panel-title {
  font-weight: 700;
  font-size: 1rem;
  margin-bottom: 0.45rem;
}

.hint {
  color: var(--muted);
  font-size: 0.9rem;
}

.result-box {
  background: linear-gradient(140deg, var(--result-bg-a) 0%, var(--result-bg-b) 100%);
  border: 1px solid #cde7e3;
  border-left: 8px solid var(--brand);
  border-radius: 16px;
  padding: 1rem;
  box-shadow: 0 10px 22px rgba(15, 118, 110, 0.12);
}

.result-box,
.result-box h3,
.result-box p,
.result-box strong {
  color: var(--result-ink) !important;
}

.badge {
  display: inline-block;
  padding: 0.2rem 0.55rem;
  border-radius: 999px;
  border: 1px solid #cbd5e1;
  background: #f8fafc;
  color: #0f172a;
  font-size: 0.8rem;
  margin: 0.12rem 0.2rem 0.12rem 0;
}

@media (prefers-color-scheme: dark) {
  .badge {
    background: #1b2638;
    border-color: #334155;
    color: #e2e8f0;
  }
}

html[data-theme="dark"] .badge,
body[data-theme="dark"] .badge,
[data-theme="dark"] .badge {
  background: #1b2638;
  border-color: #334155;
  color: #e2e8f0;
}

div[data-testid="stButton"] > button {
  border-radius: 12px;
  border: 0;
  padding: 0.65rem 0.9rem;
  font-weight: 700;
  background: linear-gradient(120deg, #0f766e 0%, #14b8a6 100%);
  color: white;
  box-shadow: 0 8px 18px rgba(20, 184, 166, 0.35);
}

div[data-testid="stButton"] > button:hover {
  transform: translateY(-1px);
}
    </style>
    """,
    unsafe_allow_html=True,
)

try:
    session = get_active_session()
except Exception:
    st.error("Snowflake session not found. Run this app inside Snowflake Streamlit.")
    st.stop()

st.sidebar.markdown("## App Notes")
st.sidebar.markdown("- Model: house_price_xgboost")
st.sidebar.markdown("- Version: v1")
st.sidebar.markdown("- Output: estimated selling price")
st.sidebar.markdown("---")
st.sidebar.markdown("### Furnishing encoding")
st.sidebar.markdown("- furnished -> 2")
st.sidebar.markdown("- semi-furnished -> 1")
st.sidebar.markdown("- unfurnished -> 0")

st.markdown(
    """
    <div class='hero'>
      <h2 style='margin:0;'>House Price Studio</h2>
      <p style='margin:0.35rem 0 0 0;'>
        Simulate a property profile and get an instant prediction from your registered Snowflake model.
      </p>
    </div>
    """,
    unsafe_allow_html=True,
)

left_col, right_col = st.columns([1.18, 0.82], gap="large")

with left_col:
    st.markdown("<div class='panel'><div class='panel-title'>Property Core</div><div class='hint'>Adjust the physical attributes first.</div></div>", unsafe_allow_html=True)
    area = st.slider("Area (m2)", min_value=50, max_value=1000, value=150, step=5)
    bedrooms = st.slider("Bedrooms", min_value=1, max_value=10, value=3)
    bathrooms = st.slider("Bathrooms", min_value=1, max_value=5, value=2)
    stories = st.slider("Stories", min_value=1, max_value=5, value=2)
    parking = st.slider("Parking spots", min_value=0, max_value=5, value=1)

    st.markdown("<div class='panel'><div class='panel-title'>Comfort and Location</div><div class='hint'>Binary features are encoded to 1 or 0 for model input.</div></div>", unsafe_allow_html=True)
    c1, c2, c3 = st.columns(3)
    with c1:
        mainroad = st.selectbox("Main road", ["yes", "no"])
        guestroom = st.selectbox("Guest room", ["yes", "no"])
        basement = st.selectbox("Basement", ["yes", "no"])
    with c2:
        hotwater = st.selectbox("Hot water heating", ["yes", "no"])
        aircon = st.selectbox("Air conditioning", ["yes", "no"])
        prefarea = st.selectbox("Preferred area", ["yes", "no"])
    with c3:
        furnished = st.selectbox("Furnishing", ["furnished", "semi-furnished", "unfurnished"])

with right_col:
    st.markdown("<div class='panel'><div class='panel-title'>Live Summary</div><div class='hint'>Snapshot of current scenario before prediction.</div></div>", unsafe_allow_html=True)

    comfort_score = sum([
        1 if mainroad == "yes" else 0,
        1 if guestroom == "yes" else 0,
        1 if basement == "yes" else 0,
        1 if hotwater == "yes" else 0,
        1 if aircon == "yes" else 0,
        1 if prefarea == "yes" else 0,
    ])

    scenario_tags = [
        f"Area {area} m2",
        f"{bedrooms} bed",
        f"{bathrooms} bath",
        f"{stories} stories",
        f"Parking {parking}",
        furnished,
    ]

    st.markdown("".join([f"<span class='badge'>{tag}</span>" for tag in scenario_tags]), unsafe_allow_html=True)
    st.progress(comfort_score / 6, text=f"Comfort score: {comfort_score}/6")

    area_band = "Compact" if area < 120 else "Family" if area < 260 else "Premium"
    st.caption(f"Property segment: {area_band}")

def encode(val):
    return 1 if val == "yes" else 0

furnish_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}

estimate_col, clear_col = st.columns([0.78, 0.22])
with estimate_col:
    run_prediction = st.button("Estimate Price", use_container_width=True)
with clear_col:
    reset_view = st.button("Reset", use_container_width=True)

if reset_view:
    st.session_state.pop("prediction_payload", None)
    st.rerun()

if run_prediction:
    input_data = pd.DataFrame([{
        "AREA": area,
        "BEDROOMS": bedrooms,
        "BATHROOMS": bathrooms,
        "STORIES": stories,
        "MAINROAD": encode(mainroad),
        "GUESTROOM": encode(guestroom),
        "BASEMENT": encode(basement),
        "HOTWATERHEATING": encode(hotwater),
        "AIRCONDITIONING": encode(aircon),
        "PARKING": parking,
        "PREFAREA": encode(prefarea),
        "FURNISHINGSTATUS": furnish_map[furnished],
    }]).astype("int16")

    with st.spinner("Running model inference..."):
        registry = Registry(session=session)
        model_ref = registry.get_model("house_price_xgboost").version("v1")
        result = model_ref.run(input_data, function_name="predict")
        price = float(result["output_feature_0"].values[0])

    st.session_state["prediction_payload"] = {
        "price": price,
        "input_data": input_data,
        "ppsm": price / max(area, 1),
    }

if "prediction_payload" in st.session_state:
    payload = st.session_state["prediction_payload"]
    price = payload["price"]
    ppsm = payload["ppsm"]
    input_data = payload["input_data"]

    st.markdown("### Prediction Output")
    st.markdown(
        f"""
        <div class='result-box'>
          <h3 style='margin:0;'>Estimated Price: {price:,.0f} EUR</h3>
          <p style='margin:0.35rem 0 0 0;'>Estimated price per m2: <strong>{ppsm:,.0f} EUR/m2</strong></p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    m1, m2, m3 = st.columns(3)
    m1.metric("Estimated price", f"{price:,.0f} EUR")
    m2.metric("Price per m2", f"{ppsm:,.0f} EUR/m2")
    m3.metric("Comfort score", f"{comfort_score}/6")

    confidence_proxy = min(max(comfort_score / 6, 0.15), 1.0)
    st.progress(confidence_proxy, text="Scenario quality indicator")

    st.markdown("### Features sent to model")
    st.dataframe(input_data, use_container_width=True)
    st.success("Inference completed and displayed successfully.")